# Introduction to GPUs in PyTorch

**Week 2 · Notebook 3 of 3 · GPUs, memory, and in-place operations**

Complete **`Week_02_intro_to_pytorch.ipynb`** and **`Week_02_python_classes_part2.ipynb`** first.

GPU stands for **graphics processing unit**. Graphics involves applying similar calculations to many pixels. For example, updating one million pixels 50 times per second means 50 million pixel updates per second.

GPUs can carry out many such calculations in parallel. The same-operation-on-many-data idea is often described as **SIMD** (single instruction, multiple data); this is a simplified description, not a requirement that every GPU core execute the same instruction at once. Large tensor operations can benefit from this parallelism too.

GPU acceleration played an important role in the development of deep learning; the [AlexNet paper](https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf) is an early example. The actual speedup depends on the operation and hardware; there is no fixed factor to expect.

Here we use PyTorch for GPU-accelerated numerical calculations, without introducing neural networks yet.


## Running this notebook in Google Colab

Select a GPU using **Runtime → Change runtime type**, choose a **GPU** hardware accelerator, and save. The particular GPU offered can vary. Start this notebook from its first code cell after changing the runtime.

This notebook needs a **CUDA-capable GPU**; it does not silently switch to CPU. The first cell checks availability and stops with a message if there is no usable CUDA GPU.

Run cells from top to bottom. The device-mismatch example and the deliberately oversized allocation are **commented out**, so they do not interrupt normal execution. The repeated memory demonstrations use moderately sized tensors so we can observe memory growth without deliberately exhausting the runtime.


In [ ]:
import torch as t

# Stop with a clear message if this runtime has no usable CUDA GPU.
assert t.cuda.is_available(), "Select a GPU runtime in Colab, then run this notebook from the top."


In [ ]:
# device='cuda' is a keyword argument: create the tensor on the CUDA GPU.
# Here CUDA is the device interface PyTorch uses for the NVIDIA GPU.
a = t.rand(3, device='cuda')
print(a)


In [ ]:
# The arithmetic and method syntax are unchanged; these operations run on the GPU.
print(2 * a)
print(a.exp())
print(a * a)


In [ ]:
# A second vector on the same GPU can be combined with a.
b = t.rand(3, device='cuda')
print(b)
print(a + b)


In [ ]:
# In this notebook, omitting device creates the tensor on the CPU.
c = t.rand(3)
# a + c  # Intentional RuntimeError: these vectors are on different devices.


In [ ]:
# .device is an attribute, so we do not add call parentheses.
print(a.device)  # cuda:0 (the first CUDA GPU).
print(b.device)  # cuda:0.
print(c.device)  # cpu.


In [ ]:
# Copy this CPU tensor to the GPU; save the returned tensor under a new name.
c_cuda = c.to(device='cuda')

# The existing tensor c is unchanged and still on the CPU.
print(c.device)
print(c_cuda.device)
print(a + c_cuda)  # Both inputs to the addition are now on the GPU.


In [ ]:
# Remove these names; later examples will reuse them for different tensors.
# We will examine how del relates to memory below.
del a
del b
del c
del c_cuda


## Why bother with GPUs? Large operations versus many small operations

Large tensor operations can use the GPU's parallel hardware efficiently. For many tiny operations, the overhead of launching each calculation can outweigh that benefit: a GPU is not automatically faster.

The following cells compare the **same calculation** on CPU and GPU, first with a large tensor and then repeatedly with a small one. Each tensor is created before timing, so these first comparisons exclude creation and transfer.

**Timing syntax.** `%%timeit` is a Colab cell command: it repeats the cell and reports execution times. It must be the first line. The GPU timing cells use `t.cuda.synchronize()` before and after the work because GPU operations are normally scheduled asynchronously; we need to wait for completion to time the calculation itself.

Compare the measured times rather than expecting a particular speedup. In the loops, `_` is an ordinary variable used by convention when we do not need its value.


In [ ]:
# Create a large CPU tensor and transfer its values to the GPU before timing.
a_cpu = t.randn(1000, 1000, 30)
a_cuda = a_cpu.to(device='cuda')


In [ ]:
%%timeit
# No dimension argument: mean() averages all entries and returns a scalar tensor.
(2 * a_cpu).mean()


In [ ]:
%%timeit
# Wait for earlier GPU work, do the calculation, then wait for it to finish.
t.cuda.synchronize()
(2 * a_cuda).mean()
t.cuda.synchronize()


In [ ]:
# Repeat the comparison with a much smaller tensor.
a_cpu = t.randn(3, 3, 30)
a_cuda = a_cpu.to(device='cuda')


In [ ]:
%%timeit
# Each timed repetition performs 100 small CPU calculations.
for _ in range(100):
    (2 * a_cpu).mean()


In [ ]:
%%timeit
# The same 100 small calculations on the GPU, including the wait for completion.
t.cuda.synchronize()
for _ in range(100):
    (2 * a_cuda).mean()
t.cuda.synchronize()


In [ ]:
del a_cpu
del a_cuda


## Gotcha 1: Initialising on the GPU versus moving to the GPU

Creating random values on the CPU and then transferring a large tensor to the GPU incurs **both** creation and transfer costs.

When we need new random values on the GPU, generating them there directly can avoid the transfer. The next cells measure these alternatives rather than assuming one is always faster.

The method `.cuda()` used below transfers a CPU tensor to the CUDA GPU, just as `.to(device='cuda')` did above. It returns the GPU tensor; it does not change the device of the original CPU tensor.


In [ ]:
%%timeit
# Generate on the GPU, then compute the mean there.
t.cuda.synchronize()
t.randn(1000, 1000, 30, device='cuda').mean()
t.cuda.synchronize()


In [ ]:
%%timeit
# Generate on the CPU, transfer to the GPU, then compute the mean on the GPU.
t.cuda.synchronize()
t.randn(1000, 1000, 30).cuda().mean()
t.cuda.synchronize()


In [ ]:
%%timeit
# Measure CPU generation on its own: one component of the preceding calculation.
t.randn(1000, 1000, 30)


In [ ]:
# For the next timing, create the CPU tensor first so only transfer is measured.
a = t.randn(1000, 1000, 30)


In [ ]:
%%timeit
# Measure the transfer of this existing CPU tensor to the GPU.
t.cuda.synchronize()
a.cuda()
t.cuda.synchronize()


In [ ]:
del a


## Gotcha 2: GPU memory

A GPU has a finite amount of memory. Requesting more than is available causes an **out-of-memory error**.

We will compare memory readings before and after allocations. `t.cuda.memory_allocated(0)` reports the bytes occupied by PyTorch tensors on GPU `0`. Dividing by `1024**3` converts bytes to **GiB** (gibibytes).

This is not a measurement of all GPU memory usage: PyTorch may keep released memory cached for reuse. Also, Colab can retain tensors displayed as a cell's result; the examples use `print` for tensor values to avoid retaining them that way.


In [ ]:
# Deliberately oversized example: 30 billion float32 entries need 120 billion bytes.
# This would exceed the memory of many Colab GPUs and raise OutOfMemoryError.
# It is retained for illustration, but left disabled: do not run it routinely.
# Do not try to avoid the GPU error by allocating this tensor on the CPU instead;
# that could exhaust the Colab runtime's RAM.
# a = t.randn(1000, 1000, 1000, 30, device='cuda')


In [ ]:
# GPU index 0 is the first GPU; ** means exponentiation in Python.
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# Allocate a sizeable tensor without requesting several gigabytes.
a = t.randn(1000, 1000, 30, device='cuda')
t.cuda.memory_allocated(0) / 1024**3
# Here there are 30 million entries. With the default float32 used here,
# each entry occupies 4 bytes: about 120 million bytes, or 0.112 GiB.


In [ ]:
# Remove the name a. With no other reference to this tensor, its allocation
# can be released for reuse by PyTorch.
del a


In [ ]:
# Check the tensor allocation again. A separate cell makes the stages easy to see;
# it is not a general requirement that del and the measurement be in different cells.
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# Suppose we only want the means, but keep every large input tensor as well.
# append(...) adds an item to the end of a Python list.
# Each input below uses about 20 million bytes; keeping 20 uses about 400 million.
# Larger tensors or more iterations would eventually exhaust GPU memory.
big_random_tensors = []
means = []
for i in range(20):
    print(t.cuda.memory_allocated(0) / 1024**3)
    big_random_tensors.append(t.randn(100, 100, 500, device='cuda'))
    means.append(big_random_tensors[i].mean())


In [ ]:
# The list still holds all the input tensors. Index 2 selects the third one.
print(big_random_tensors[2].shape)


In [ ]:
# We only needed the means. Remove both lists before trying the next approach.
del big_random_tensors
del means


In [ ]:
# With those references removed, their tensor allocations can be released.
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# Keep the means, but no list of large input tensors.
means = []
for i in range(20):
    print(t.cuda.memory_allocated(0) / 1024**3)
    big_random_tensor = t.randn(100, 100, 500, device='cuda')
    means.append(big_random_tensor.mean())


In [ ]:
# Each reassignment replaces the reference to the previous large input.
# These readings stay roughly level, apart from the small saved scalar means.
# While the next tensor is being created, old and new tensors can briefly coexist.
# But after the loop, one large tensor is still accessible:
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# A for-loop does not give this variable its own local scope.
# The name still refers to the input from the last iteration.
print(big_random_tensor.shape)


In [ ]:
# Remove that last reference to the large input.
del big_random_tensor


In [ ]:
# The large input is gone, but the scalar tensors in means are still stored.
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# Another approach: make the large input local to a function call.
means = []


def inner():
    big_random_tensor = t.randn(100, 100, 500, device='cuda')
    # This appends to the existing list means; it does not reassign the name means.
    means.append(big_random_tensor.mean())


for i in range(20):
    print(t.cuda.memory_allocated(0) / 1024**3)
    inner()
# Each call finishes without keeping a reference to its large input tensor.


In [ ]:
# No large input remains accessible outside inner(); only the scalar means remain.
t.cuda.memory_allocated(0) / 1024**3


## References and memory release

As with the lists in Week 1, **assignment does not make a copy**. Two names can refer to the same tensor. This matters both when we modify it and when we try to release its memory.


In [ ]:
# Start with a small CPU tensor so that the values are easy to inspect.
a = t.ones(3)
a[0] = 2  # Replace the entry at index 0.
print(a)


In [ ]:
# b is another name for the SAME tensor, not for a copy.
b = a
b[1] = 3  # Change that tensor through the name b.
print(b)
print(a)  # The change is visible through a too.


In [ ]:
# Assign a new GPU tensor to a. The old small CPU tensor still has the name b.
a = t.randn(100, 100, 500, device='cuda')
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# Now make b refer to that same GPU tensor. This does not allocate another copy.
b = a
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# del removes this NAME, not every other reference to the object.
del a
print(b.shape)  # We can still access the GPU tensor through b.


In [ ]:
# The tensor's allocation is still present because b refers to it.
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# Remove the remaining name referring to that large tensor.
del b


In [ ]:
# Its allocation can now be released; the means saved earlier are separate tensors.
t.cuda.memory_allocated(0) / 1024**3


## In-place operations

**In place** means modifying an existing tensor rather than making a new tensor for the result.

For example, `b = a + 100` creates a new result tensor and leaves `a` unchanged:

$$b_{ijk} = a_{ijk} + 100.$$

If both tensors remain accessible, we store two full arrays. If the old values are no longer needed, we can instead update `a` in place.

A trailing underscore identifies many in-place tensor methods: `a.add_(100)` adds `100` to the existing entries of `a`. The name of the method is `add_`, including the underscore.

Some functions also accept an **`out` keyword argument** specifying an existing tensor in which to write the result. In `t.matmul(B, C, out=A)` below, the result of `B @ C` is written into `A`; `B` and `C` are not modified. This illustrates choosing a destination, not overwriting an input.

**For later:** in-place changes can interfere with gradient computation if they overwrite values that are still needed. Our examples here do not compute gradients; do not assume that every such update is safe inside a neural network.


In [ ]:
A = t.zeros(3, 3)  # Existing destination tensor.
B = t.randn(3, 3)
C = t.randn(3, 3)
t.matmul(B, C, out=A)  # Compute B @ C and store the result in A.
print(A)


Now compare the GPU memory needed for a new result with an in-place update. Both examples shift standard normal random values to have mean `100` and standard deviation `1`.

In the printed examples, `a[0, 0, 0]` selects one entry. **`.item()`** converts that one-element tensor to an ordinary Python number for display; it does not change `a`.


In [ ]:
# Store both the original random values and a separate shifted result.
a = t.randn(100, 100, 500, device='cuda')
print(f"a[0,0,0] = {a[0, 0, 0].item()}")
b = a + 100
print(f"b[0,0,0] = {b[0, 0, 0].item()}")
# Two full tensors are now retained: about 40 million bytes in total.
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
del a
del b


In [ ]:
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
# Instead, overwrite the original entries with their shifted values.
a = t.randn(100, 100, 500, device='cuda')
print(f"a[0,0,0] = {a[0, 0, 0].item()}")
a.add_(100)
print(f"a[0,0,0] = {a[0, 0, 0].item()}")
# Only one full tensor is retained: about 20 million bytes.
t.cuda.memory_allocated(0) / 1024**3


In [ ]:
del a


In [ ]:
# The final small baseline includes the scalar means saved earlier.
t.cuda.memory_allocated(0) / 1024**3
